<html><div style="font-size:7pt">This notebook may contain text, code and images generated by artificial intelligence. Used model: claude-sonnet-4-20250514, vision model: claude-sonnet-4-20250514, endpoint: None, bia-bob version: 0.34.3.. It is good scientific practice to check the code and results it produces carefully. <a href="https://github.com/haesleinhuepf/bia-bob">Read more about code generation using bia-bob</a></div></html>

# Image Embeddings and UMAP Generation

This notebook demonstrates how to:
1. Generate vision embeddings using the [openai/clip-vit-base-patch32](https://huggingface.co/openai/clip-vit-base-patch32)
2. Create a 3D UMAP visualization of the embeddings
3. Save the results for VEST

In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel
from datasets import load_dataset
from umap import UMAP
import random
from pathlib import Path
import requests
from io import BytesIO
import torch

In [2]:
import transformers
transformers.__version__

'4.56.1'

In [3]:
base_dir = Path("./")
images_dir = base_dir / "images"

## Load Vision Embedding Model

In [4]:
from transformers import CLIPProcessor, CLIPModel
import pandas as pd
import torch
from PIL import Image
import requests
import os

# Initialize models
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

## Generate Vision Embeddings for All Images

In [5]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F

# Get list of all PNG files in the images folder and subfolders
png_files = []
for root, dirs, files in os.walk(images_dir):
    for f in files:
        if f.lower().endswith('.png'):
            png_files.append(os.path.join(root, f))

# Initialize lists to store results
filenames = []
embeddings = []
images = []

print(f"Processing {len(png_files)} images for embeddings...")

# Loop through all PNG files
for i, image_path in enumerate(png_files):
    # Obtain filename from path
    filename = os.path.relpath(image_path, images_dir)
    
    # Load the image
    current_image = Image.open(image_path)
    images.append(np.asarray(current_image))

    # Apply the processing pipeline
    current_inputs = clip_processor(images=current_image, return_tensors="pt")
    with torch.no_grad():
        current_np_emb = clip_model.get_image_features(**current_inputs).squeeze().tolist()

    # Store results
    filenames.append(filename)
    embeddings.append(current_np_emb)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(png_files)} images")



# Create DataFrame
df = pd.DataFrame({
    'filename': filenames,
    'embedding': embeddings
})

print(f"Successfully processed {len(df)} images")
display(df.head())

Processing 1000 images for embeddings...
Processed 100/1000 images
Processed 200/1000 images
Processed 300/1000 images
Processed 400/1000 images
Processed 500/1000 images
Processed 600/1000 images
Processed 700/1000 images
Processed 800/1000 images
Processed 900/1000 images
Processed 1000/1000 images
Successfully processed 1000 images


,filename,embedding
0,CHAMMI-75_small\hpa0018\0d8a5134-bb9c-11e8-b2b...,"[0.43723195791244507, -0.0828917920589447, 0.1..."
1,CHAMMI-75_small\hpa0018\0e56e5cc-bbbb-11e8-b2b...,"[0.18899831175804138, -0.05855303257703781, 0...."
2,CHAMMI-75_small\hpa0018\1040_117_H4_3-ch2.png,"[0.2645382881164551, -0.39633893966674805, 0.4..."
3,CHAMMI-75_small\hpa0018\11356_96_G2_1-ch0.png,"[0.42666202783584595, 0.26250573992729187, 0.0..."
4,CHAMMI-75_small\hpa0018\12867_104_B4_2-ch3.png,"[0.3678473234176636, 0.13091140985488892, 0.03..."


## Create 3D UMAP Visualization from Embeddings

In [6]:
# Convert embeddings list to numpy array matrix
embedding_matrix = np.stack(df['embedding'].values)

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print("Creating 3D UMAP coordinates...")

# Create 3D UMAP
umap_reducer = UMAP(n_components=3, random_state=42)
umap_coords_actual = umap_reducer.fit_transform(embedding_matrix)

# Add UMAP coordinates to dataframe
df['x'] = umap_coords_actual[:, 0]
df['y'] = umap_coords_actual[:, 1]
df['z'] = umap_coords_actual[:, 2]

print("UMAP coordinates created successfully")
print(f"X range: {df['x'].min():.2f} to {df['x'].max():.2f}")
print(f"Y range: {df['y'].min():.2f} to {df['y'].max():.2f}")
print(f"Z range: {df['z'].min():.2f} to {df['z'].max():.2f}")
display(df.head())

Embedding matrix shape: (1000, 512)
Creating 3D UMAP coordinates...


C:\Users\rober\miniforge3\envs\bob-env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP coordinates created successfully
X range: 7.11 to 13.09
Y range: 6.59 to 12.82
Z range: -1.76 to 3.09


,filename,embedding,x,y,z
0,CHAMMI-75_small\hpa0018\0d8a5134-bb9c-11e8-b2b...,"[0.43723195791244507, -0.0828917920589447, 0.1...",12.636723,11.331880,-1.214043
1,CHAMMI-75_small\hpa0018\0e56e5cc-bbbb-11e8-b2b...,"[0.18899831175804138, -0.05855303257703781, 0....",10.252423,11.194191,1.422441
2,CHAMMI-75_small\hpa0018\1040_117_H4_3-ch2.png,"[0.2645382881164551, -0.39633893966674805, 0.4...",9.979407,8.270468,-1.579605
3,CHAMMI-75_small\hpa0018\11356_96_G2_1-ch0.png,"[0.42666202783584595, 0.26250573992729187, 0.0...",12.252362,10.393790,2.149001
4,CHAMMI-75_small\hpa0018\12867_104_B4_2-ch3.png,"[0.3678473234176636, 0.13091140985488892, 0.03...",12.881889,9.993117,0.562215


## Save Results to CSV File

In [7]:
# Save the dataframe to CSV
output_path = base_dir / "data.csv"

# Convert embedding arrays to strings for CSV storage
df_to_save = df.copy()
df_to_save['embedding'] = df_to_save['embedding'].apply(lambda x: ','.join(map(str, x)))

df_to_save.to_csv(output_path, index=False)

print(f"Results saved to {output_path}")
print(f"Final dataframe shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
display(df[['filename', 'x', 'y', 'z']].head(10))

Results saved to data.csv
Final dataframe shape: (1000, 5)
Columns: ['filename', 'embedding', 'x', 'y', 'z']


,filename,x,y,z
0,CHAMMI-75_small\hpa0018\0d8a5134-bb9c-11e8-b2b...,12.636723,11.331880,-1.214043
1,CHAMMI-75_small\hpa0018\0e56e5cc-bbbb-11e8-b2b...,10.252423,11.194191,1.422441
2,CHAMMI-75_small\hpa0018\1040_117_H4_3-ch2.png,9.979407,8.270468,-1.579605
3,CHAMMI-75_small\hpa0018\11356_96_G2_1-ch0.png,12.252362,10.393790,2.149001
4,CHAMMI-75_small\hpa0018\12867_104_B4_2-ch3.png,12.881889,9.993117,0.562215
5,CHAMMI-75_small\hpa0018\14118_1553_B5_2-ch0.png,12.546753,10.508176,2.098821
6,CHAMMI-75_small\hpa0018\17037_621_G5_2-ch2.png,12.641100,11.377781,-1.260635
7,CHAMMI-75_small\hpa0018\17872_1272_B5_2-ch2.png,12.708707,11.264095,-1.076555
8,CHAMMI-75_small\hpa0018\19025_1033_F7_1-ch1.png,7.255277,12.288476,2.249349
9,CHAMMI-75_small\hpa0018\19316_618_E5_4-ch2.png,12.624886,11.361588,-1.203741
